[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jibby2k1/SPS_Curriculum/blob/main/Intro_Math/Hilbert_Spaces/Hilbert_Spaces.ipynb)


Content for and by IEEE Signal Processing Society. (Raul Valle & Contributors)

# Hilbert Spaces & Fourier, Properly

> ⚠️ **Draft — pending instructor review.** Visuals execute; proofs need a human pass before teaching. Remove this banner after review.

The bridge between [Measure Theory's](../Analysis/Measure_Theory.ipynb) $L^2$ and [DSP's](../../Intro_DSP/README.md) transforms: once you see $L^2$ as a geometry — with angles, projections, and orthonormal bases — 'transform = change of basis' stops being a slogan and becomes a theorem, and Parseval becomes Pythagoras.

## 1. Pre-requisites

- [Linear Algebra](../Linear_Algebra/Linear_Algebra.ipynb) S1–S2 (bases, projections in $\mathbb{R}^N$).
- [Measure Theory](../Analysis/Measure_Theory.ipynb) S3 ($L^p$ spaces) for full rigor — skimmable if you accept $L^2$ as 'finite-energy signals'.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

---
### 🕐 Session 1 of 3 — *Inner Products & Orthonormal Systems* (~35 min)
**Goal:** give function spaces a geometry; measure angles between signals.
**Builds on:** [Linear Algebra](../Linear_Algebra/Linear_Algebra.ipynb) S1–S2. &nbsp; **Feeds into:** Session 2 (the projection theorem).

---

<details>
<summary>🎓 <b>Teacher notes — Session 1: Inner Products & Orthonormal Systems</b></summary>

**Timing (~35 min).** 10 min the "dot product with infinitely many coordinates" picture · 10 min Cauchy–Schwarz with its proof · 8 min the correlation demo · 7 min completeness and what $L^2$ is. Cauchy–Schwarz is the anchor; everything else can be compressed.

**Board first — the whole session in one move.** Write $x^\top y = \sum_i x_i y_i$ above $\langle f, g\rangle = \int f\bar g$ and say the integral is the sum with the index made continuous. Every subsequent definition is then a translation exercise rather than a new idea: length is energy, angle is correlation, orthogonality is zero correlation. Students who get this analogy stop treating function spaces as exotic.

**Motivate the Cauchy–Schwarz proof — do not just present it.** The discriminant trick reads as a rabbit from a hat, and students dislike it for good reason. Motivate it: we are asking how small $\|f - tg\|$ can get as we scale $g$, which is exactly "how much of $f$ can $g$ explain." Expand, notice it is a quadratic in $t$ that can never be negative because it is a norm squared, and *therefore* the discriminant cannot be positive. Framed that way the trick is forced rather than clever, and the minimising $t$ is the projection coefficient of Session 2 — worth flagging forward.

**Misconception.** "Orthogonal means unrelated." It means *uncorrelated at zero lag*, which is much weaker. The 90°-shifted cosine in the demo is the same frequency and perfectly predictable from the sine, yet the inner product is zero. Have someone explain the difference before you move on — this misconception resurfaces in [Statistical SP](../../Intro_DSP/Statistical_Signal_Processing.ipynb) as independence versus uncorrelatedness.

**Ask the room.** "Why do we need *completeness* — what could go wrong without it?" Analogy: the rationals have holes, so a sequence of rationals can converge to $\sqrt2$ and escape the space. Session 2's proof breaks at exactly that point, so plant the idea here and pay it off there. Students who have done [Sequences & Series](../Analysis/Numerical_Sequences_and_Series.ipynb) will make the connection themselves if you prompt.

**Pacing note.** If the room is strong, state Hölder and note Cauchy–Schwarz is the $p=q=2$ case. If not, skip it — it is a remark, not a dependency.
</details>

## 2. Inner Product Spaces

💡 **Intuition.** An inner product is a *dot product with the finite dimensions removed*: $\langle f, g \rangle = \int f \bar{g}$ sums the pointwise agreement of two signals just as $x^Ty$ sums coordinate agreement. Everything geometric follows: length $\|f\| = \sqrt{\langle f, f\rangle}$ (the energy!), angle via Cauchy–Schwarz, orthogonality as zero correlation. A **Hilbert space** is an inner product space that is also *complete* — Cauchy sequences of signals converge ([Sequences & Series](../Analysis/Numerical_Sequences_and_Series.ipynb) upgraded to functions) — and $L^2$ is the star example.

**Cauchy–Schwarz.** $|\langle f, g \rangle| \le \|f\| \|g\|$.

*Proof.* For any $t \in \mathbb{R}$ (real case): $0 \le \|f - t g\|^2 = \|f\|^2 - 2t\langle f, g\rangle + t^2 \|g\|^2$ — a quadratic in $t$ that never goes negative, so its discriminant $4\langle f,g\rangle^2 - 4\|f\|^2\|g\|^2 \le 0$. $\blacksquare$

This is Hölder with $p = q = 2$, and it's why 'correlation coefficient' lands in $[-1, 1]$ — matched filtering in [Statistical SP](../../Intro_DSP/Statistical_Signal_Processing.ipynb) is maximizing this very inner product.

In [2]:
# Angles between signals: correlation as cosine
t = np.linspace(0, 1, 1000)
f = np.sin(2 * np.pi * 5 * t)
for name, g in [("same sine", np.sin(2*np.pi*5*t)), ("shifted 90°", np.cos(2*np.pi*5*t)),
                ("different freq", np.sin(2*np.pi*8*t)), ("sine + noise", np.sin(2*np.pi*5*t) + np.random.default_rng(0).standard_normal(1000))]:
    cos = (f @ g) / (np.linalg.norm(f) * np.linalg.norm(g))
    print(f"cos∠(f, {name:15s}) = {cos:+.3f}")
print("→ orthogonality = zero correlation; harmonics are mutually orthogonal")

cos∠(f, same sine      ) = +1.000
cos∠(f, shifted 90°    ) = +0.000
cos∠(f, different freq ) = +0.000
cos∠(f, sine + noise   ) = +0.611
→ orthogonality = zero correlation; harmonics are mutually orthogonal


**What just happened.** Four signals, four angles, and Cauchy–Schwarz is what guarantees the numbers all landed in $[-1, 1]$ — the cosine of an angle between two *functions* is a well-defined thing, and this is it.

Read the rows in order, because each is a different lesson:

- **Same sine, $+1.000$.** Angle zero, the signals are parallel. A signal is perfectly correlated with itself, which is the trivial case that fixes the scale.
- **Shifted 90°, $+0.000$.** This is the row to dwell on. A sine and a cosine of the *same frequency* are orthogonal, yet either one determines the other completely. Orthogonality is not independence, and not unrelatedness — it is zero correlation *at this particular alignment*. This single fact is why quadrature modulation can send two independent data streams on one carrier: I and Q are orthogonal, so the receiver separates them with an inner product.
- **Different frequency, $+0.000$.** Distinct harmonics are mutually orthogonal — the property that makes the Fourier basis a *basis* in Session 3, and the reason a coefficient can be extracted by one inner product without solving a linear system.
- **Sine + noise, $+0.611$.** Correlation degrades gracefully rather than collapsing: the noise is orthogonal to the sine on average, so it contributes to $\|g\|$ but not to $\langle f, g\rangle$. Matched filtering is exactly this observation turned into a detector — project onto the template you are hunting for, and noise energy spreads across all the directions you are not looking in.

Everything after this is bookkeeping on top of one idea: an inner product turns a space of signals into a geometry, and geometric intuition transfers wholesale from $\mathbb{R}^3$.

---
### 🕐 Session 2 of 3 — *The Projection Theorem* (~35 min)
**Goal:** prove that closed subspaces admit unique best approximations; get least squares for signals.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (Fourier bases).

---

<details>
<summary>🎓 <b>Teacher notes — Session 2: The Projection Theorem</b></summary>

**Timing (~35 min).** 8 min the shadow picture and why infinite dimensions are harder · 15 min the proof · 5 min the Bessel/coefficient payoff · 7 min the square-wave demo. The proof is the session; do not let the demo eat it.

**Board first.** Draw the plane-and-point picture from [Linear Algebra S2](../Linear_Algebra/Linear_Algebra.ipynb) and get the room to agree the nearest point exists and the error is perpendicular — in $\mathbb{R}^3$ this is obvious. Then say: in infinite dimensions, *existence is the hard part*, and the entire proof below is about earning something you just took for granted. That reframing is what makes a dry proof feel necessary.

**The crux of the proof — slow down here.** Students follow the parallelogram-law algebra and still miss the point of it, so name it explicitly: we do not construct $\hat f$, we take a *minimising sequence* and show it is Cauchy. The parallelogram law is the tool that converts "these two are both nearly optimal" into "these two are near *each other*". Then completeness produces a limit, and closedness of $M$ keeps that limit inside $M$. Ask which hypothesis each step consumed — it is the cleanest example in the curriculum of a proof where every assumption is load-bearing.

**Where it would fail.** Worth 60 seconds: give a non-closed subspace, such as finite-support sequences inside $\ell^2$, and note the minimising sequence can run off to a limit outside the set. This is the payoff for the "rationals have holes" analogy planted in Session 1, and it is what "Hilbert" buys over "inner product space."

**Misconception.** "The best approximation is found by solving a big least-squares system." For an *orthonormal* system it is not — each coefficient is one independent inner product, $\langle f, e_k\rangle$, with no matrix to invert and no interaction between coefficients. That independence is precisely what orthogonality buys, and it is why adding a harmonic never forces you to recompute the ones you already had. Contrast with a non-orthogonal basis, where every coefficient changes.

**Ask the room.** Before running the demo: "we add more harmonics — does the overshoot at the jump get smaller?" Nearly everyone says yes. It does not, and the debrief works through why. This is the best "your intuition is wrong, and the theorem still holds" moment available in the workshop, so do not spoil it early.
</details>

## 3. Best Approximation

💡 **Intuition.** Same shadow picture as [Linear Algebra S2](../Linear_Algebra/Linear_Algebra.ipynb) — but in infinite dimensions the *existence* of the closest point is no longer free: you need completeness to stop minimizing sequences from converging to a hole. That's the real job of the 'Hilbert' in Hilbert space.

**Theorem (projection).** Let $M$ be a closed subspace of a Hilbert space $H$, $f \in H$. Then there is a *unique* $\hat{f} \in M$ minimizing $\|f - m\|$ over $m \in M$, characterized by $f - \hat{f} \perp M$.

*Proof sketch.* Take a minimizing sequence $m_n$ with $\|f - m_n\| \to d = \inf$. The **parallelogram law** $\|a+b\|^2 + \|a-b\|^2 = 2\|a\|^2 + 2\|b\|^2$ applied to $a = f - m_n$, $b = f - m_k$ shows $\|m_n - m_k\|^2 \le 2\|f-m_n\|^2 + 2\|f-m_k\|^2 - 4d^2 \to 0$ (using $\frac{m_n + m_k}{2} \in M$) — the sequence is Cauchy, and **completeness** hands us the limit $\hat f \in M$ ($M$ closed). Orthogonality: if $\langle f - \hat f, m\rangle \ne 0$ for some unit $m \in M$, then $\hat f + \langle f - \hat f, m\rangle m$ is strictly closer — contradiction. Uniqueness follows from Pythagoras. $\blacksquare$

**Payoff.** For an orthonormal system $\{e_k\}$, the projection onto $\mathrm{span}\{e_1..e_K\}$ is $\hat f = \sum_{k\le K} \langle f, e_k\rangle e_k$ — *keep the top coefficients* — with error $\|f\|^2 - \sum_{k \le K} |\langle f, e_k\rangle|^2$ (**Bessel**: partial sums of coefficient energy never exceed the signal's energy). Every truncated Fourier/wavelet approximation in DSP is this theorem running.

In [3]:
# Best L² approximation of a square wave by K harmonics — projection in action
t = np.linspace(0, 1, 4000, endpoint=False)
sq = np.sign(np.sin(2 * np.pi * t))

plt.figure(figsize=(8.5, 3))
plt.plot(t, sq, "k", linewidth=1, label="square wave")
for K, alpha in [(1, 0.5), (5, 0.7), (25, 1.0)]:
    approx = np.zeros_like(t)
    for k in range(1, K + 1, 2):                       # odd harmonics only
        approx += (4 / (np.pi * k)) * np.sin(2 * np.pi * k * t)
    plt.plot(t, approx, alpha=alpha, label=f"K={K} harmonics")
plt.legend(); plt.title("Projections onto growing subspaces (note Gibbs' stubborn ears)")
plt.tight_layout(); plt.show()

/tmp/ipykernel_2016431/1099153454.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**What just happened.** Three projections of the same square wave onto three nested subspaces. Each curve is the *provably optimal* $L^2$ approximation using the harmonics available to it — not a fit, not an optimisation we ran, but the projection theorem evaluated as a formula. And because the subspaces are nested, adding harmonics only ever improves the approximation: the $K=1$ coefficients are untouched in the $K=25$ curve, which is the orthogonality payoff from the previous cell made visible.

**Now the ears.** The overshoot at each jump — Gibbs' phenomenon — does not shrink as $K$ grows. It narrows, sliding closer to the discontinuity, but its height stubbornly holds at about 9% of the jump no matter how many harmonics you add. Students almost always predict it will decay, and the fact that it does not looks at first like a contradiction of "best approximation."

It is not a contradiction, and resolving it is the sharpest lesson in the session: **the theorem promises convergence in $\|\cdot\|_2$, not pointwise.** The overshoot keeps its height while its *width* shrinks toward zero, so the energy in the error — an integral of height² × width — goes to zero exactly as promised. $\|f - \hat f\|_2 \to 0$ and $\hat f(t) \not\to f(t)$ uniformly are perfectly compatible statements, and the square wave is the standard example proving they are different.

That distinction is why $L^2$ is the right home for signals. It is a weaker notion of "close" than pointwise agreement, and that weakness is a feature: it is stable under the operations engineers actually perform, it is what makes energy the currency of the theory, and it is what will let Parseval be an exact equality in Session 3. When a DSP text says a truncated Fourier series "converges to" a square wave, this is the sense meant — and the ringing you see on a filtered digital edge is this plot, in hardware.

---
### 🕐 Session 3 of 3 — *Orthonormal Bases & Parseval* (~40 min)
**Goal:** define completeness of a basis; prove Parseval; see why the Fourier basis is special.
**Builds on:** Session 2.

---

<details>
<summary>🎓 <b>Teacher notes — Session 3: Orthonormal Bases & Parseval</b></summary>

**Timing (~40 min).** 8 min completeness, Bessel → Parseval · 10 min Parseval as Pythagoras · 12 min the eigenfunction argument (the payoff of the whole workshop) · 7 min the demo · 3 min buffer.

**Board first.** Draw a right triangle, write $c^2 = a^2 + b^2$, then extend it to three perpendicular axes, then say "now let the number of axes be countably infinite." Parseval is that sentence. Presenting it as a new theorem about Fourier transforms hides the fact that students have known it since secondary school.

**Bessel versus Parseval — the one distinction to nail.** Bessel is an *inequality* and holds for any orthonormal system; Parseval is *equality* and holds precisely when the system is complete. The gap between them is the energy living in directions your system cannot see. Concrete version: keep only the even-indexed basis vectors and you still get Bessel, but half the energy vanishes. Completeness is exactly the claim that nothing is missed, and it is the hard part — Riesz–Fischer is stated here, not proved, which is worth saying out loud so nobody thinks a step was skipped by accident.

**The most important idea in the workshop.** Not Parseval — the *eigenfunction* argument. There are infinitely many orthonormal bases of $L^2$, so "we could expand in any basis" raises the obvious question of why this one runs all of DSP. The answer: complex exponentials are the eigenfunctions of every LTI system, so the Fourier basis simultaneously diagonalises *every* convolution operator at once. Convolution becoming multiplication is not a computational trick, it is diagonalisation — the same phenomenon as [Linear Algebra S3](../Linear_Algebra/Linear_Algebra.ipynb)'s eigenvectors, in function space. Give this its full 12 minutes; it is the sentence that reframes the students' entire DSP background.

**Ask the room.** "Wavelets also form an orthonormal basis of $L^2$ — so why isn't DSP built on them instead?" Because they do not diagonalise convolution; they buy time localisation and give up the eigenfunction property. That trade is the subject of [Foundations 2](../../Intro_DSP/Foundations_of_Signal_Processing_2.ipynb), and framing it as a trade rather than a ranking sets that workshop up well.

**Point at the standard deviation.** In the demo output, `std 1.90e-14` is the number to project on screen. It is floating-point zero, which means the eigenfunction claim is not approximately true here — it is exactly true, and only rounding separates us from it. Students respond to that far more strongly than to the algebra.

**Draft status.** This notebook still carries the ⚠️ review banner; the proofs need a human pass before the sessions are taught or recorded. Leave the banner in place until that review happens.
</details>

## 4. Complete Orthonormal Bases

An orthonormal system $\{e_k\}$ is a **basis** (complete) if finite combinations are dense — equivalently, if Bessel's inequality is *equality* for every $f$:

$$\|f\|^2 = \sum_k |\langle f, e_k \rangle|^2 \qquad \textbf{(Parseval)}$$

💡 **Intuition.** Parseval is *Pythagoras with infinitely many perpendicular directions*: energy in the signal = summed energy of its coordinates, because orthogonal components can't interfere. 'The DFT/Fourier transform preserves energy' is not a happy accident — it is the statement that complex exponentials form a complete orthonormal basis of $L^2$ (Riesz–Fischer; completeness proof via Stone–Weierstrass or Fejér, stated not proved).

**Why the Fourier basis, of all bases?** The exponentials $e^{i\omega t}$ are the **eigenfunctions of time-invariant systems**: feed $e^{i\omega t}$ into any LTI filter and you get $H(\omega) e^{i\omega t}$ — same signal, scaled ([Linear Algebra S3](../Linear_Algebra/Linear_Algebra.ipynb)'s eigenvectors, in function space). The Fourier basis simultaneously diagonalizes *every* convolution — which is why convolution becomes multiplication and DSP lives in the frequency domain.

In [4]:
# Parseval, numerically, with the DFT (unitary convention)
rng = np.random.default_rng(1)
x = rng.standard_normal(1024)
X = np.fft.fft(x) / np.sqrt(1024)
print(f"time-domain energy   {np.sum(x**2):.6f}")
print(f"coefficient energy   {np.sum(np.abs(X)**2):.6f}   (Parseval: equal)")

# And the eigenfunction property: convolution acts diagonally on exponentials
h = np.exp(-np.arange(30) / 5.0)                       # some LTI filter
k = 37                                                  # pick a frequency bin
e = np.exp(2j * np.pi * k * np.arange(1024) / 1024)    # basis exponential
y = np.convolve(e, h)[:1024]                            # (circular edge effects negligible mid-signal)
ratio = y[200:800] / e[200:800]
print(f"filter output / input on e_k: constant ≈ {ratio.mean():.4f} (std {ratio.std():.2e}) = H(ω_k)")

time-domain energy   1009.508255
coefficient energy   1009.508255   (Parseval: equal)
filter output / input on e_k: constant ≈ 2.6988-2.4525j (std 1.90e-14) = H(ω_k)


**What just happened.** Two claims, two measurements, and the second is the one worth remembering.

**Parseval, exactly.** Time-domain energy `1009.508255`, coefficient energy `1009.508255` — agreement to every digit printed. Not approximately, not up to a constant: the DFT in its unitary convention is a unitary map on $\mathbb{C}^{1024}$ — the complex analogue of a rotation — and unitary maps preserve length. That is the whole content of "the Fourier transform preserves energy," and it is why the factor of $1/\sqrt{N}$ matters — the more common conventions park that constant somewhere else and Parseval picks up a scale factor. If a transform ever fails to conserve energy, suspect the normalisation before the mathematics.

**The eigenfunction property, to machine precision.** Feed a basis exponential through an arbitrary LTI filter and the output is the *same* exponential, scaled: the output-to-input ratio is constant across 600 samples with a standard deviation of **1.90e-14**. That is floating-point zero. The filter did not reshape $e_k$ at all — it multiplied it by the single complex number $H(\omega_k) \approx 2.699 - 2.453j$, whose magnitude is the gain at that frequency and whose argument is the phase shift.

**Why this is the punchline of the workshop.** $L^2$ has infinitely many orthonormal bases, and Parseval holds in every one of them, so energy conservation alone cannot explain why DSP is built on this particular basis. The eigenfunction property can. Complex exponentials are the eigenvectors of *every* convolution operator simultaneously, so moving to the Fourier basis diagonalises every LTI system at once — and a diagonal operator is just pointwise multiplication. "Convolution in time is multiplication in frequency" is therefore not a computational coincidence to memorise; it is the statement that we changed to the basis where these operators are diagonal, exactly as one diagonalises a matrix in linear algebra.

Every frequency response plot the students have ever read is a list of eigenvalues, and the `1.90e-14` above is the receipt.

## 5. Conclusion

$L^2$ is geometry: Cauchy–Schwarz gives angles, completeness + the parallelogram law give unique best approximations, and Parseval is Pythagoras. The Fourier basis earns its throne by diagonalizing every LTI system at once.

---
## Where next

- [Foundations of Signal Processing](../../Intro_DSP/Foundations_of_Signal_Processing_1.ipynb) — re-read Sessions 3–5 with this geometry in mind.
- [Kernel Methods & RKHS](../../Intro_Mach_Learn/Kernel_Methods.ipynb) — Hilbert spaces where evaluation is an inner product: the ML sequel.
- [Foundations of Signal Processing 2](../../Intro_DSP/Foundations_of_Signal_Processing_2.ipynb) — wavelets: a *different* orthonormal basis with different trade-offs.